# first Title 
## Second Title
- item 1
- item 2
- item 3
**abcde**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df = pd.read_csv("RFM_Raw.csv")
df.head()

,Date,InvoNo,CustomerKey,Amount
0,20180413,AP1804933,A1255,9997.02
1,20180423,AP1807826,A1188,9997.02
2,20180424,AP1800872,A0846,9997.02
3,20180426,AP1803170,A1204,9997.02
4,20180501,MA1806692,A0907,9997.02


In [15]:
import pandas as pd

# 假設你的原始資料長這樣：
# data = {
#     'CustomerKey': ['C001', 'C001', 'C002', 'C003', 'C003'],
#     'Date': ['2023-01-01', '2023-01-10', '2023-01-05', '2023-01-02', '2023-01-08'],
#     '发票编號': ['INV001', 'INV002', 'INV003', 'INV004', 'INV005'],
#     'Amount': [100, 200, 150, 300, 500]
# }
# df = pd.DataFrame(data)

# 1. 確保「Date」欄位是 datetime 格式，以便進行Date運算
df['Date'] = pd.to_datetime(df['Date'])

# 2. 找出整個資料集中的「最大Date」（作為基準點）
analysis_date = df['Date'].max() + pd.Timedelta(days=1)

# 3. 使用 groupby 進行聚合運算
rfm_df = (
    df.groupby('CustomerKey')
    .agg(
        # F: InvoNo的計數 (可以用 count 或 nunique，這裡用 count 計算總發票筆數)
        F=('InvoNo', 'count'),
        # M: Amount加總
        M=('Amount', 'sum'),
        # R: 找出該客戶的最大Date
        max_client_date=('Date', 'max'),
    )
    .reset_index()
)

# 4. 計算 R：基準最大Date 與 該客戶最大交易Date的差額
# .dt.days 可以將時間差轉換為整數的天數
rfm_df['R'] = (analysis_date - rfm_df['max_client_date']).dt.days

# 5. 重新命名欄位並調整順序，符合你要求的四個欄位：Customer, F, M, R
rfm_df = rfm_df.rename(columns={'CustomerKey': 'Customer'})

# 移除中間步驟不需要的輔助欄位 (max_client_date)
rfm_df = rfm_df[['Customer', 'F', 'M', 'R']]

# 檢視結果
print(rfm_df)

    Customer   F            M  R
0      A0350  25  51238.11440  1
1      A0351   6  17890.31200  1
2      A0352  14  46424.03852  1
3      A0353  14  27607.68636  1
4      A0354   6   9973.92064  1
..       ...  ..          ... ..
727    A1413  23  52738.79456  1
728    A1415  20  53019.49664  1
729    A1416  24  49938.38378  1
730    A1419  22  64257.84520  1
731    A1420  14  25054.18342  1

[732 rows x 4 columns]


In [18]:
import pandas as pd

# 假設 df 是你的原始 DataFrame
# 為了程式碼對應，先將欄位重新命名（若你的欄位名稱本來就是英文，可略過此步驟）
df = df.rename(
    columns={
        "客戶ID": "CustomerKey",
        "日期": "Date",
        "發票編號": "InvoNo",
        "金額": "Amount",
    }
)

# 1. 確保日期欄位為 datetime 型態，以便後續計算天數差
df["Date"] = pd.to_datetime(df["Date"])

# 2. 設定基準日 (R 的計算需要「與客戶交易日期最大值的差額」，通常是以資料集中的最大日期作為基準)
# 如果你是要跟「今天」比，可以改用 pd.Timestamp('today')
max_dataset_date = df["Date"].max()

# 3. 進行 groupby 與聚合運算
rfm_df = (
    df.groupby("CustomerKey")
    .agg(
        F=(
            "InvoNo",
            "count",
        ),  # F: InvoNo 計數 (如果要求不重複計數，可以改用 'nunique')
        M=("Amount", "sum"),  # M: Amount 加總
        Max_Date=("Date", "max"),  # 先找出該客戶的最大日期
    )
    .reset_index()
)

# 4. 計算 R：基準日與該客戶最大交易日期的差額（以天數表示）
rfm_df["R"] = (max_dataset_date - rfm_df["Max_Date"]).dt.days

# 5. 整理欄位順序與名稱，符合你的需求：Customer, F, M, R
rfm_df = rfm_df.rename(columns={"CustomerKey": "Customer"})
rfm_df = rfm_df[["Customer", "F", "M", "R"]]

# 檢視結果
print(rfm_df)

    Customer   F            M  R
0      A0350  25  51238.11440  0
1      A0351   6  17890.31200  0
2      A0352  14  46424.03852  0
3      A0353  14  27607.68636  0
4      A0354   6   9973.92064  0
..       ...  ..          ... ..
727    A1413  23  52738.79456  0
728    A1415  20  53019.49664  0
729    A1416  24  49938.38378  0
730    A1419  22  64257.84520  0
731    A1420  14  25054.18342  0

[732 rows x 4 columns]


In [20]:
import pandas as pd

# 確保欄位名稱正確
df = df.rename(
    columns={
        "客戶ID": "CustomerKey",
        "日期": "Date",
        "發票編號": "InvoNo",
        "金額": "Amount",
    }
)

# 【修改這裡】明確指定格式為 YYYYMMDD
df["Date"] = pd.to_datetime(df["Date"].astype(str), format="%Y%m%d")

# 檢查看看是否轉型成功
print("修正後最早日期：", df["Date"].min())
print("修正後最晚日期：", df["Date"].max())

ValueError: time data "1970-01-01 00:00:00.020180413" doesn't match format "%Y%m%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.